In [1]:
import numpy as np
import pandas as p
import os

In [2]:
def load_merged_df(attack_path):
    dfs = {}
    trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

    for item in os.listdir(attack_path):
        item_path = os.path.join(attack_path, item)
        if os.path.isdir(item_path):
            scores = p.read_csv(item_path + '/cosine/voxceleb1_scores_cal.csv')
            merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')
            dfs[item] = merged_df

    return dfs
        

In [3]:
def get_threshold(prior):
    return -np.log(prior) + np.log(1-prior)

In [4]:
def get_results(dfs):
    count = 0
    count_tar = 0

    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            key = (row.modelid, row.segmentid)

            if key not in results:
                    results[key] = []

            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

            if(row.targettype == 'target'):
                 count_tar = count_tar + 1

            count = count + 1

    print(count_tar)
    return results

In [5]:
def get_non_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'nontarget'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [6]:
def get_non_tar_clean(merged_df):
    result = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'nontarget'):
            
            key = (row.modelid, row.segmentid)

            if key not in result:
                result[key] = []
            
            score_key = (row.targettype, row.LLR)
            result[key].append(score_key)

    return result

In [7]:
def get_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'target'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [8]:
def get_tar_clean(merged_df):
    results = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'target'):
            
            key = (row.modelid, row.segmentid)

            if key not in results:
                results[key] = []
            
            score_key = (row.targettype, row.LLR)
            results[key].append(score_key)

    return results

In [9]:
def get_nb_imposters(results, t):
    passed = 0

    for key in results:
        for y in results[key]:
            if(y[1] > t):
                passed = passed + 1
                break

    return passed

In [10]:
def get_frr(results, t):
    rejected = 0
    total = 0

    for key in results:

        for y in results[key]:
            total = total + 1
            if(y[1] < t):
                rejected = rejected + 1
                #break

    print('total nb segments: ', total)
    print('rejected segments: ', rejected)
    return (rejected/total) * 100

In [11]:
def get_asr(result, passed):
    return (passed/len(result)) * 100

ATTACK 20 CLUSTERS 1.2

POISONED

In [12]:
attack='exp/scores/attack_20_clusters_1.2/triggers'
merged_dfs = load_merged_df(attack)

In [21]:
t = get_threshold(0.5)

results_non_tar = get_non_tar_results(merged_dfs)
results_tar = get_tar_results(merged_dfs)
imposters = get_nb_imposters(results_non_tar, t)

In [22]:
imposters

7501

In [23]:
get_asr(results_non_tar, imposters)

14.925878022087355

In [20]:
get_frr(results_tar, t)

total nb segments:  994900
rejected segments:  982402


98.74379334606493

CLEAN

In [16]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_20_clusters_1.1'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [17]:
t = get_threshold(0.5)

result_non_tar_clean = get_non_tar_clean(merged_df)
results_tar_clean = get_tar_clean(merged_df)
imposters_clean = get_nb_imposters(result_non_tar_clean, t)

In [18]:
get_asr(result_non_tar_clean, imposters_clean)

10.381056611282458

In [19]:
get_frr(results_tar_clean, t)

total nb segments:  49745
rejected segments:  3778


7.594733139008945